# 03 · SALT-GNN 학습과 A3 우선 평가

02에서 선택한 장치의 입력 검증을 통과한 후 실행합니다. 기본 worker는
**A3 6에폭·전체 Monitor → A3 전체 Val → 체크포인트·Val 고정 → A3 전체 Test → A0·A1·A2 학습 → 전체 Val 비교·고정 → 나머지 Test**를 수행합니다.
완료한 A3 Val/Test는 재사용합니다. Test 결과를 이후 모델 선택·튜닝에 사용하지 않습니다.

| 항목 | 기본값 |
|---|---|
| 그래프·피처 준비 | CUDA 탐색 + GPU 상주 거래 피처, 일별 노드/degree 갱신 |
| 배치 | 논리 8,192 / micro 최대 4,096 / 간선 분할 기준 6,000,000 |
| GPU 메모리 한도 | 90% (V100 32GB에서 측정한 기본값) |
| 학습 | 모델별 6에폭, seed 42, Adam, LR 0.0012, 양성 손실 가중치 7, clip 5 |
| 1에폭 | Train 전체 87,572,226건; 같은 조건으로 처음부터 학습하면 10,714회 업데이트 |
| 입력·모델 | 노드 20 / 간선 24차원, 2층, hidden 20, heads 2 |
| 체크포인트 | 전체 Monitor F1@0.5 최대, 동률은 앞선 에폭 |

아래 실행 설정 표는 00에서 실제 생성한 프로토콜을 읽습니다. 변경한 값이 있으면 그 값이 적용됩니다.
백그라운드 worker의 최근 배치 속도와 진행률을 3초마다 갱신할 수 있습니다.
노트북 커널의 표시를 중단해도 worker는 유지됩니다. 학습 중단은 별도 중단 셀에서 요청합니다.
새 실행은 처음부터 현재 배치를 사용하며, 기존 서버 A3의 배치 변경 이력을 이어받지 않습니다.


In [ ]:
from pathlib import Path
import json
import sys
import subprocess

PROJECT_DIR = Path.cwd().resolve()
link = PROJECT_DIR / ".salt_project.json"
if not link.exists():
    raise FileNotFoundError("이 노트북들이 있는 폴더에서 00_환경설정과_실행준비.ipynb를 먼저 실행하세요.")
project = json.loads(link.read_text())
WORK_DIR = Path(project["work_dir"])
DEVICE = project["device"]
for module_name in ("salt_daily", "runtime", "run", "notebook_control"):
    loaded = sys.modules.get(module_name)
    location = getattr(loaded, "__file__", None)
    if location and not Path(location).resolve().is_relative_to(WORK_DIR):
        raise RuntimeError("다른 WORK_DIR가 import되어 있습니다. 커널을 재시작하세요.")
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

def command(stage, device=DEVICE):
    subprocess.run([sys.executable, "-u", str(WORK_DIR/"run.py"), stage, "--device", device], cwd=WORK_DIR, check=True)

print("작업 폴더:", WORK_DIR)


In [ ]:
from IPython.display import display, Markdown
if (WORK_DIR/"protocol.json").exists():
    p = json.loads((WORK_DIR/"protocol.json").read_text())
    t = p["training"]
    rows = [("graph backend", p["graph"].get("sampler_backend")),
            ("input backend", p.get("input_pipeline", {}).get("backend")),
            ("logical / micro", f"{t['logical_batch']} / {t['microbatch_max']}"),
            ("edge split", f"{t['max_edges_per_microbatch']:,}"),
            ("GPU memory fraction", t["cuda_memory_fraction"]),
            ("epochs / seed", f"{t['epochs_primary']} / {t['seed']}"),
            ("LR", t["learning_rates_primary"]),
            ("updates per full epoch", t["updates_per_full_epoch"])]
    display(Markdown("|설정|적용값|\n|---|---|\n" + "\n".join(f"|{k}|{v}|" for k,v in rows)))
else:
    print("02에서 학습 입력을 준비하면 실제 설정 표가 표시됩니다.")


### 이 단계의 실행 코드

아래 소스는 00 노트북이 읽어 작업 폴더에 실행 파일로 만듭니다. 코드를 수정하면 노트북을 저장하고 **새 WORK_DIR**로 00을 다시 실행하세요.

In [ ]:
SOURCES = {}


In [ ]:
# salt_daily/model.py
SOURCES['salt_daily/model.py'] = '''"""SALT with paper-listed hyperparameters and the retained daily V10 graph adapter.

All variants share the input dimensions, output head, sampled contexts and
two message-passing layers. This is not certified as an official reproduction.
"""
from . import config  # Establish the existing dependency paths first.
import torch
from torch import nn
from torch_geometric.nn import TransformerConv


def message_statistics(messages, destination, nodes):
    """Population mean/std/min/max, with exact zeros for empty neighborhoods."""
    width = messages.shape[1]
    counts = messages.new_zeros(nodes)
    counts.index_add_(0, destination, messages.new_ones(len(messages)))
    total = messages.new_zeros((nodes, width)).index_add(0, destination, messages)
    mean = total / counts.clamp_min(1).unsqueeze(1)
    # Centered second moment avoids cancellation for nearly equal messages.
    centered = messages - mean[destination]
    variance = messages.new_zeros((nodes, width)).index_add(0, destination, centered.square())
    variance = variance / counts.clamp_min(1).unsqueeze(1)
    std = torch.where(variance > 0, variance.clamp_min(1e-20).sqrt(), torch.zeros_like(variance))
    index = destination[:, None].expand(-1, width)
    low = messages.new_full((nodes, width), float('inf')).scatter_reduce(0, index, messages, reduce='amin', include_self=True)
    high = messages.new_full((nodes, width), float('-inf')).scatter_reduce(0, index, messages, reduce='amax', include_self=True)
    occupied = counts[:, None] > 0
    low = torch.where(occupied, low, torch.zeros_like(low))
    high = torch.where(occupied, high, torch.zeros_like(high))
    return torch.cat((mean, std, low, high), dim=1), counts


def scaled_statistics(statistics, degree, delta):
    gamma = degree.log1p().unsqueeze(1)
    amplification = gamma / delta
    attenuation = torch.where(gamma > 0, delta / gamma.clamp_min(1e-12), torch.zeros_like(gamma))
    return torch.cat((statistics, statistics * amplification, statistics * attenuation), dim=1)


def mlp(input_width, hidden):
    return nn.Sequential(nn.Linear(input_width, hidden), nn.ReLU(), nn.Linear(hidden, hidden))


def model_kwargs(protocol, delta):
    architecture = protocol.get('architecture', {})
    assert architecture.get('layers', 2) == 2, 'This implementation has two message-passing layers'
    return dict(hidden=architecture.get('common_hidden', 20),
                heads=architecture.get('heads', 2), delta=delta,
                dropout=architecture.get('dropout', .083),
                final_dropout=architecture.get('final_dropout', .288),
                edge_dim=architecture.get('edge_dim', 14))


class Statistics(nn.Module):
    def __init__(self, hidden, edge_dim):
        super().__init__()
        self.edge = nn.Linear(edge_dim, hidden)
        self.update = mlp(13 * hidden, hidden)

    def forward(self, nodes, index, attributes, degree, delta):
        messages = nodes[index[0]] + self.edge(attributes)
        statistics, _ = message_statistics(messages, index[1], len(nodes))
        return self.update(torch.cat((nodes, scaled_statistics(statistics, degree, delta)), dim=1))


class DailyLayer(nn.Module):
    def __init__(self, hidden, heads, mode, edge_dim=14, dropout=.083):
        super().__init__()
        self.mode = mode
        self.attention = nn.ModuleList([
            TransformerConv(hidden, hidden // heads, heads=heads, edge_dim=edge_dim,
                            dropout=0., root_weight=False, bias=False) for _ in range(2)
        ]) if mode in ('A0', 'A3') else None
        self.statistics = nn.ModuleList([Statistics(hidden, edge_dim) for _ in range(2)]) if mode in ('A1', 'A3') else None
        if self.attention is not None:
            for layer in self.attention:
                # PyG creates this projection even with root_weight=False.
                # The root path belongs exclusively to the common fusion MLP.
                layer.lin_skip.requires_grad_(False)
        self.attention_merge = nn.Linear(2 * hidden, hidden) if self.attention is not None else None
        self.statistics_merge = nn.Linear(2 * hidden, hidden) if self.statistics is not None else None
        self.fusion = mlp((3 if mode == 'A3' else 2) * hidden, hidden)
        # The paper gives a dropout rate without a complete executable placement.
        # Use hidden-state dropout after fusion; attention coefficient dropout stays 0.
        self.dropout = nn.Dropout(dropout)

    def forward(self, nodes, index, attributes, incoming, outgoing, degree, delta):
        parts = [nodes]
        masks = (incoming, outgoing)
        edges = (index[:, incoming], index[:, outgoing].flip(0))
        if self.statistics is not None:
            stats = [layer(nodes, edges[i], attributes[masks[i]], degree[:, i], delta[i])
                     for i, layer in enumerate(self.statistics)]
            parts.append(self.statistics_merge(torch.cat(stats, dim=1)))
        if self.attention is not None:
            attention = [layer(nodes, edges[i], attributes[masks[i]]) for i, layer in enumerate(self.attention)]
            parts.append(self.attention_merge(torch.cat(attention, dim=1)))
        return self.dropout((nodes + self.fusion(torch.cat(parts, dim=1))) / 2)


class DailySALT(nn.Module):
    def __init__(self, variant='A3', hidden=20, heads=2, delta=(1., 1.), dropout=.083, final_dropout=.288, edge_dim=14):
        super().__init__()
        if variant not in ('A0', 'A1', 'A2', 'A3'):
            raise ValueError(variant)
        self.variant = variant
        self.register_buffer('delta', torch.tensor(delta, dtype=torch.float32))
        self.node_encoder = nn.Linear(20, hidden)  # 19 business features + ego indicator.
        if variant == 'A2':
            self.attention_layers = nn.ModuleList([DailyLayer(hidden, heads, 'A0', edge_dim=edge_dim, dropout=dropout) for _ in range(2)])
            self.statistic_layers = nn.ModuleList([DailyLayer(hidden, heads, 'A1', edge_dim=edge_dim, dropout=dropout) for _ in range(2)])
            self.final_fusion = mlp(3 * hidden, hidden)
        else:
            self.layers = nn.ModuleList([DailyLayer(hidden, heads, variant, edge_dim=edge_dim, dropout=dropout) for _ in range(2)])
        self.head = nn.Sequential(nn.Linear(2 * hidden + edge_dim, hidden), nn.ReLU(),
                                  nn.Dropout(final_dropout), nn.Linear(hidden, 2))

    def forward(self, x, index, attributes, target_index, target_attributes, incoming, outgoing, degree, return_logits=False):
        initial = self.node_encoder(x).relu()
        if self.variant == 'A2':
            attention = statistics = initial
            for a_layer, s_layer in zip(self.attention_layers, self.statistic_layers):
                attention = a_layer(attention, index, attributes, incoming, outgoing, degree, self.delta)
                statistics = s_layer(statistics, index, attributes, incoming, outgoing, degree, self.delta)
            nodes = (initial + self.final_fusion(torch.cat((initial, statistics, attention), dim=1))) / 2
        else:
            nodes = initial
            for layer in self.layers:
                nodes = layer(nodes, index, attributes, incoming, outgoing, degree, self.delta)
        logits = self.head(torch.cat((nodes[target_index[0]], nodes[target_index[1]], target_attributes), dim=1))
        # sigmoid(logit_1 - logit_0) equals the two-class softmax fraud probability.
        return logits if return_logits else logits[:, 1] - logits[:, 0]

'''


In [ ]:
# salt_daily/training.py
SOURCES['salt_daily/training.py'] = '''"""Full daily epochs and full unweighted evaluation, with atomic resume."""
import random
import signal
import time

import numpy as np
import torch
import torch.nn.functional as F

from .config import ROOT, RUN
from .storage import read_json, write_json, sha, log, utc, memory
from .data import cohort, daily_orders, date, tensors, cleanup
from .model import DailySALT, model_kwargs
from .metrics import summarize
from .evaluation import score


class StopRequested(Exception):
    pass


class StopSignals:
    def __enter__(self):
        self.requested = False
        def stop(signum, frame):
            self.requested = True
        self.handlers = {sig: signal.signal(sig, stop) for sig in (signal.SIGINT, signal.SIGTERM)}
        return self

    def check(self):
        if self.requested:
            raise StopRequested('Stopped at a committed training/evaluation boundary')

    def __exit__(self, *args):
        for sig, handler in self.handlers.items():
            signal.signal(sig, handler)


def save_checkpoint(path, state):
    temporary = path.with_suffix('.tmp')
    torch.save(state, temporary)
    temporary.replace(path)


def loss_denominator(labels, positive_weight=7):
    labels = np.asarray(labels) > 0
    return float(len(labels) + (positive_weight-1)*labels.sum())


def microbatch_loss(logits, labels, logical_denominator, positive_weight=7):
    """Sum parts against ONE logical denominator, matching weighted CE mean."""
    return F.cross_entropy(logits, labels, weight=logits.new_tensor([1., positive_weight]),
                           reduction='sum') / logical_denominator


def resource_guard(folder, variant, boundary):
    from pathlib import Path
    value = memory()
    if isinstance(value.get('memory.max'), int) and Path('/sys/fs/cgroup/memory.stat').exists():
        stats = dict(line.split() for line in Path('/sys/fs/cgroup/memory.stat').read_text().splitlines())
        if int(stats['anon']) > value['memory.max'] - 1024**3:
            raise RuntimeError(f'Insufficient anonymous-memory headroom at {variant} {boundary}; resume checkpoint')


def expected_epoch_updates(train, state, batch_size):
    """Keep committed old-batch updates and count the exact remaining batches."""
    spans = train.meta['spans']
    cursor, offset = state['day_cursor'], state['logical_cursor']
    assert 0 <= cursor <= len(spans)
    if cursor == len(spans):
        assert offset == 0
    else:
        assert 0 <= offset <= spans[cursor]['stop']-spans[cursor]['start']
    completed = sum(s['stop']-s['start'] for s in spans[:cursor]) + offset
    assert completed == state['epoch_targets'], 'Resume cursor disagrees with committed target count'
    remaining = [s['stop']-s['start']-(offset if i == cursor else 0)
                 for i, s in enumerate(spans) if i >= cursor]
    return state['epoch_updates'] + sum((n+batch_size-1)//batch_size for n in remaining)


def train_variant(variant, protocol, loader, device='cuda'):
    folder = RUN/'models'/variant
    folder.mkdir(parents=True, exist_ok=True)
    protocol_hash = sha(ROOT/'protocol.json')
    signature = read_json(ROOT/'code_manifest.json')
    if (folder/'complete.json').exists():
        done = read_json(folder/'complete.json')
        assert done['protocol_sha256'] == protocol_hash and done['code_manifest'] == signature
        assert done['checkpoint_sha256'] == sha(folder/'best.pt')
        return
    seed = protocol['training']['seed']
    torch.set_num_threads(1); torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    if device == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.cuda.set_per_process_memory_fraction(protocol['training'].get('cuda_memory_fraction', .9))
    delta = read_json(ROOT/'calibration.json')['delta']
    model = DailySALT(variant, **model_kwargs(protocol, delta)).to(device)
    optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad),
                                 lr=protocol['training']['learning_rates_primary'][0])
    train, monitor = cohort('train_gnn'), cohort('monitor')
    config = protocol['training']
    epochs, batch_size = config['epochs_primary'], config['logical_batch']
    state = dict(epoch=1, day_cursor=0, logical_cursor=0, phase='train', history=[],
                 best={'F1': -1., 'AP': -1., 'epoch': 0}, epoch_targets=0, epoch_updates=0,
                 loss_sum=0., loss_weight_sum=0., epoch_train_seconds=0.)
    if (folder/'last.pt').exists():
        saved = torch.load(folder/'last.pt', map_location='cpu', weights_only=False)
        assert saved['protocol_sha256'] == protocol_hash and saved['code_manifest'] == signature
        assert saved['population_ipw'] is False
        model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
        state = saved['progress']; torch.set_rng_state(saved['rng_cpu'])
        if device == 'cuda':
            torch.cuda.set_rng_state_all(saved['rng_cuda'])
        del saved
    def save():
        save_checkpoint(folder/'last.pt', dict(model=model.state_dict(), optimizer=optimizer.state_dict(),
            progress=state, protocol_sha256=protocol_hash, code_manifest=signature, population_ipw=False,
            rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all() if device == 'cuda' else []))
    log(folder, 'training_started', variant=variant, epoch=state['epoch'], phase=state['phase'],
        day_cursor=state['day_cursor'], logical_cursor=state['logical_cursor'], requested=state['epoch_targets'],
        total=len(train), full_epoch=True, population_ipw=False)
    try:
        with StopSignals() as stop:
            while state['epoch'] <= epochs:
                epoch = state['epoch']
                for group in optimizer.param_groups:
                    group['lr'] = config['learning_rates_primary'][epoch-1]
                if state['phase'] == 'train':
                    expected_updates = expected_epoch_updates(train, state, batch_size)
                    model.train()
                    day_start = state['day_cursor']
                    orders = daily_orders(train, epoch, start_day_cursor=day_start)
                    for day_cursor, (day, order) in enumerate(orders, start=day_start):
                        stop.check(); loader.set_day(day)
                        offset = state['logical_cursor'] if day_cursor == state['day_cursor'] else 0
                        for start in range(offset, len(order), batch_size):
                            stop.check(); resource_guard(folder, variant, 'before_logical_batch')
                            begin = time.monotonic(); logical = order[start:start+batch_size]
                            denominator = loss_denominator(train['y'][logical], config['positive_weight'])
                            optimizer.zero_grad(set_to_none=True)
                            loss_value = 0.; actual = max_nodes = max_edges = 0
                            microbatches_used = max_microbatch_targets = 0
                            prepare_seconds = model_seconds = 0.
                            previous = time.monotonic()
                            if device == 'cuda': torch.cuda.reset_peak_memory_stats()
                            for take, arrays, metadata in loader.microbatches(train, logical,
                                    microbatch=config['microbatch_max'], max_edges=config['max_edges_per_microbatch']):
                                ready = time.monotonic(); prepare_seconds += ready-previous
                                batch = tensors(arrays, device)
                                labels = torch.as_tensor(train['y'][take] > 0, dtype=torch.long, device=device)
                                logits = model(*batch, return_logits=True)
                                assert logits.shape == (len(labels), 2) and torch.isfinite(logits).all()
                                loss = microbatch_loss(logits, labels, denominator, config['positive_weight'])
                                assert torch.isfinite(loss)
                                loss.backward(); loss_value += float(loss.detach()); actual += len(take)
                                microbatches_used += 1; max_microbatch_targets = max(max_microbatch_targets, len(take))
                                max_nodes = max(max_nodes, metadata['nodes']); max_edges = max(max_edges, metadata['edges'])
                                del batch, labels, logits, loss, arrays
                                previous = time.monotonic(); model_seconds += previous-ready
                            assert actual == len(logical)
                            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), config['gradient_clip'], error_if_nonfinite=True)
                            optimizer.step()
                            state['epoch_targets'] += actual; state['epoch_updates'] += 1
                            state['loss_sum'] += loss_value*denominator; state['loss_weight_sum'] += denominator
                            state['epoch_train_seconds'] += time.monotonic()-begin
                            state['day_cursor'] = day_cursor; state['logical_cursor'] = start+actual
                            save(); cleanup()
                            batch_wall_seconds = time.monotonic()-begin
                            log(folder, 'training_progress', variant=variant, epoch=epoch, day=date(day),
                                phase='train', requested=state['epoch_targets'], scored=state['epoch_targets'], total=len(train),
                                optimizer_updates=state['epoch_updates'], loss=loss_value, gradient_norm=float(norm),
                                train_seconds=state['epoch_train_seconds'], max_microbatch_nodes=max_nodes,
                                max_microbatch_edges=max_edges, microbatches_used=microbatches_used,
                                max_microbatch_targets=max_microbatch_targets, memory=memory(),
                                batch_targets=actual, batch_wall_seconds=batch_wall_seconds,
                                prepare_seconds=prepare_seconds, model_seconds=model_seconds,
                                cuda_peak_allocated_bytes=torch.cuda.max_memory_allocated() if device == 'cuda' else 0,
                                cuda_allocated_bytes=torch.cuda.memory_allocated() if device == 'cuda' else 0,
                                cuda_reserved_bytes=torch.cuda.memory_reserved() if device == 'cuda' else 0)
                            stop.check()
                        state['day_cursor'] = day_cursor+1; state['logical_cursor'] = 0; save()
                    assert state['epoch_targets'] == len(train)
                    assert state['epoch_updates'] == expected_updates
                    state['phase'] = 'monitor'; save()
                scores, seconds = score(model, monitor, loader, device, folder, 'monitor', epoch, stop.check)
                log(folder, 'aggregating_metrics', variant=variant, scope='monitor', epoch=epoch,
                    scored=len(monitor), requested=len(monitor))
                metrics = summarize(monitor['y'], scores, monitor['ids'], detailed=False)
                metrics.update(epoch=epoch, train_requested=state['epoch_targets'], train_scored=state['epoch_targets'],
                    optimizer_updates=state['epoch_updates'], train_seconds=state['epoch_train_seconds'],
                    monitor_seconds=seconds, mean_loss=state['loss_sum']/state['loss_weight_sum'])
                if metrics['F1'] > state['best']['F1']:
                    state['best'] = dict(F1=metrics['F1'], AP=metrics['AP'], epoch=epoch)
                    save_checkpoint(folder/'best.pt', dict(model=model.state_dict(), variant=variant, epoch=epoch,
                        protocol_sha256=protocol_hash, code_manifest=signature, monitor=metrics, delta=delta,
                        population_ipw=False))
                state['history'].append(metrics); write_json(folder/'history.json', state['history'])
                log(folder, 'epoch_complete', variant=variant, best=state['best'], **metrics)
                state.update(epoch=epoch+1, phase='train', day_cursor=0, logical_cursor=0, epoch_targets=0,
                             epoch_updates=0, loss_sum=0., loss_weight_sum=0., epoch_train_seconds=0.)
                save(); stop.check()
        write_json(folder/'complete.json', dict(status='complete', epochs=epochs, best=state['best'],
            protocol_sha256=protocol_hash, code_manifest=signature, checkpoint_sha256=sha(folder/'best.pt'),
            coverage=1., completed_utc=utc(), total_training_targets=len(train)*epochs))
        log(folder, 'training_complete', variant=variant, best=state['best'])
    except BaseException as error:
        log(folder, 'interrupted' if isinstance(error, StopRequested) else 'training_error',
            variant=variant, error_type=type(error).__name__, detail=str(error)[:700])
        raise
    finally:
        del model, optimizer; cleanup()
        if device == 'cuda':
            torch.cuda.empty_cache()


def require_trained_variant(protocol, variant):
    """Final evaluation uses a completed Train/Monitor run, never a partial epoch."""
    if variant not in protocol['variants']:
        raise ValueError(f'Unknown variant: {variant}')
    folder = RUN/'models'/variant
    if not (folder/'complete.json').exists():
        raise RuntimeError(f'{variant}: finish all training epochs and Monitor before final Val/Test')
    done = read_json(folder/'complete.json')
    assert done['status'] == 'complete' and done['epochs'] == protocol['training']['epochs_primary']
    assert done['protocol_sha256'] == sha(ROOT/'protocol.json')
    assert done['code_manifest'] == read_json(ROOT/'code_manifest.json')
    assert done['checkpoint_sha256'] == sha(folder/'best.pt'), 'Completed checkpoint changed'
    return done


def _checked_validation(protocol, variant, validation):
    done = require_trained_variant(protocol, variant)
    folder = RUN/'models'/variant
    assert validation == read_json(folder/'validation.json'), 'Validation record changed'
    assert validation['variant'] == variant and validation['scope'] == 'validation'
    assert validation['requested'] == validation['scored'] == len(cohort('validation'))
    assert validation['coverage'] == 1.
    assert validation['protocol_sha256'] == sha(ROOT/'protocol.json')
    assert validation['code_manifest'] == read_json(ROOT/'code_manifest.json')
    assert validation['checkpoint_sha256'] == done['checkpoint_sha256']
    return done


def _variant_freeze_record(protocol, variant, validation):
    done = _checked_validation(protocol, variant, validation)
    folder = RUN/'models'/variant
    return dict(scope='variant', variant=variant, protocol_sha256=sha(ROOT/'protocol.json'),
                code_manifest=read_json(ROOT/'code_manifest.json'),
                checkpoint_sha256=done['checkpoint_sha256'],
                training_complete_sha256=sha(folder/'complete.json'),
                validation_sha256=sha(folder/'validation.json'),
                checkpoint_selection='full Monitor F1 at fixed probability 0.5; earliest epoch tie',
                test_policy='this variant only; checkpoint and full Val locked before its Test')


def freeze_variant(protocol, variant, validation):
    """Freeze one finished model without requiring any other model's checkpoint."""
    item = _variant_freeze_record(protocol, variant, validation)
    path = RUN/'models'/variant/'evaluation_frozen.json'
    if path.exists():
        assert read_json(path) == item, 'Frozen variant evaluation changed'
    else:
        write_json(path, item)
    return item


def _test_authorization(protocol, variant):
    """Check local/global freezes before opening the Test population or scores."""
    folder = RUN/'models'/variant
    local = folder/'evaluation_frozen.json'
    global_path = RUN/'selection_frozen.json'
    if not local.exists() and not global_path.exists():
        raise RuntimeError(f'{variant}: freeze the checkpoint and full Val before Test')
    if local.exists():
        expected = _variant_freeze_record(protocol, variant, read_json(folder/'validation.json'))
        assert read_json(local) == expected, 'Frozen variant evaluation changed'
    if global_path.exists():
        frozen = read_json(global_path)
        assert frozen['protocol_sha256'] == sha(ROOT/'protocol.json')
        assert frozen['code_manifest'] == read_json(ROOT/'code_manifest.json')
        assert frozen['checkpoints'][variant] == sha(folder/'best.pt')
        assert frozen['validation_files'][variant] == sha(folder/'validation.json')
    path = local if local.exists() else global_path
    return dict(scope='variant' if local.exists() else 'global', sha256=sha(path))


def evaluate_scope(protocol, loader, scope, device='cuda', variants=None):
    assert scope in ('validation', 'test')
    selected = list(protocol['variants'] if variants is None else variants)
    assert selected and len(selected) == len(set(selected)), 'Empty/duplicate evaluation variants'
    freezes = {}
    for variant in selected:
        require_trained_variant(protocol, variant)
        if scope == 'test':
            freezes[variant] = _test_authorization(protocol, variant)
    if device == 'cuda':
        torch.cuda.set_per_process_memory_fraction(protocol['training'].get('cuda_memory_fraction', .9))
    data = cohort(scope); results = {}
    with StopSignals() as stop:
        for variant in selected:
            stop.check(); folder = RUN/'models'/variant
            checkpoint_sha = sha(folder/'best.pt')
            if (folder/f'{scope}.json').exists():
                result = read_json(folder/f'{scope}.json')
                assert result['variant'] == variant and result['scope'] == scope
                assert result['checkpoint_sha256'] == checkpoint_sha
                assert result['protocol_sha256'] == sha(ROOT/'protocol.json')
                assert result['code_manifest'] == read_json(ROOT/'code_manifest.json')
                assert result['requested'] == result['scored'] == len(data) and result['coverage'] == 1.
                if scope == 'test':
                    assert result['test_authorization'] == freezes[variant]
                results[variant] = result
                continue
            saved = torch.load(folder/'best.pt', map_location='cpu', weights_only=False)
            assert saved['protocol_sha256'] == sha(ROOT/'protocol.json')
            assert saved['code_manifest'] == read_json(ROOT/'code_manifest.json')
            assert saved['population_ipw'] is False
            model = DailySALT(variant, **model_kwargs(protocol, saved['delta'])).to(device)
            model.load_state_dict(saved['model'])
            scores, seconds = score(model, data, loader, device, folder, scope, saved['epoch'], stop.check)
            log(folder, 'aggregating_metrics', variant=variant, scope=scope,
                epoch=saved['epoch'], scored=len(data), requested=len(data))
            result = summarize(data['y'], scores, data['ids'])
            daily = []
            for span in data.meta['spans']:
                section = slice(span['start'], span['stop'])
                if span['start'] != span['stop']:
                    daily.append(dict(day=date(span['day']), **summarize(data['y'][section],
                                      scores[section], data['ids'][section], detailed=False)))
            result.update(variant=variant, scope=scope, checkpoint_epoch=saved['epoch'],
                checkpoint_sha256=checkpoint_sha, protocol_sha256=sha(ROOT/'protocol.json'),
                code_manifest=read_json(ROOT/'code_manifest.json'),
                threshold_policy='fixed_probability_0.5', seconds=seconds, daily=daily,
                test_evaluated=scope == 'test')
            if scope == 'test':
                result['test_authorization'] = freezes[variant]
            write_json(folder/f'{scope}.json', result); results[variant] = result
            log(folder, f'{scope}_complete', variant=variant, scored=len(data), requested=len(data))
            del model, saved, scores; cleanup()
            if device == 'cuda':
                torch.cuda.empty_cache()
    return results


def evaluate_variant(protocol, loader, variant='A3', device='cuda'):
    """Full Val -> immutable local freeze -> full Test for one completed model."""
    validation = evaluate_scope(protocol, loader, 'validation', device, variants=[variant])[variant]
    frozen = freeze_variant(protocol, variant, validation)
    test = evaluate_scope(protocol, loader, 'test', device, variants=[variant])[variant]
    folder = RUN/'models'/variant
    item = dict(status='complete', variant=variant, protocol_sha256=sha(ROOT/'protocol.json'),
                code_manifest=read_json(ROOT/'code_manifest.json'),
                checkpoint_sha256=frozen['checkpoint_sha256'],
                validation_sha256=sha(folder/'validation.json'), test_sha256=sha(folder/'test.json'),
                freeze_sha256=sha(folder/'evaluation_frozen.json'),
                validation_scored=validation['scored'], test_scored=test['scored'], full_population_certified=True)
    path = folder/'evaluation_complete.json'
    if path.exists():
        assert read_json(path) == item, 'Completed variant evaluation changed'
    else:
        write_json(path, item)
    log(folder, 'variant_evaluation_complete', variant=variant, validation_scored=validation['scored'], test_scored=test['scored'])
    return dict(validation=validation, frozen=frozen, test=test)


def freeze_selection(protocol, validation):
    """Freeze the overall winner using full Val only; early local Test is reusable."""
    for variant in protocol['variants']:
        _checked_validation(protocol, variant, validation[variant])
        if (RUN/'models'/variant/'evaluation_frozen.json').exists():
            _test_authorization(protocol, variant)
    selected = max(protocol['variants'], key=lambda v: validation[v]['F1'])
    item = dict(protocol_sha256=sha(ROOT/'protocol.json'),
                code_manifest=read_json(ROOT/'code_manifest.json'), selected_variant=selected,
                selection='full Val F1 at fixed 0.5; variant order breaks ties; Test never used',
                checkpoints={v: sha(RUN/'models'/v/'best.pt') for v in protocol['variants']},
                validation_files={v: sha(RUN/'models'/v/'validation.json') for v in protocol['variants']})
    path = RUN/'selection_frozen.json'
    if path.exists():
        assert read_json(path) == item, 'Frozen selection changed'
    else:
        write_json(path, item)
    return item
'''


In [ ]:
# notebook_control.py
SOURCES['notebook_control.py'] = '''"""Manual notebook control and full-population progress gauges."""
import fcntl
import json
import os
from pathlib import Path
import signal
import subprocess
import sys
import time
from html import escape

from salt_daily.config import ROOT, RUN
from salt_daily.storage import read_json, write_json

_CHILDREN = {}


def read(path, default):
    return read_json(path) if path.exists() else default


def identity(pid):
    try:
        proc = Path('/proc')/str(int(pid))
        stat = (proc/'stat').read_text().split(') ', 1)[1].split()
        argv = (proc/'cmdline').read_bytes().decode().split('\\0')
        index = argv.index(str(ROOT/'run.py'))
        if stat[0] == 'Z' or argv[index+1] not in ('run', 'train', 'evaluate') or (proc/'cwd').resolve() != ROOT:
            return None
        return dict(pid=int(pid), start_ticks=int(stat[19]))
    except (OSError, ValueError, IndexError, TypeError):
        return None


def worker():
    item = read(RUN/'notebook_launch.json', {})
    current = identity(item.get('pid'))
    if current and current.get('start_ticks') == item.get('start_ticks'):
        return current
    for proc in Path('/proc').iterdir():
        if proc.name.isdigit():
            current = identity(proc.name)
            if current:
                return current
    return None


def start_or_attach(device='cuda', phase='run', variant=None):
    if phase not in ('run', 'train', 'evaluate'):
        raise ValueError('phase must be run, train or evaluate')
    if variant is not None and (phase != 'evaluate' or variant not in ('A3','A0','A1','A2')):
        raise ValueError('variant requires an individual evaluation')
    if device not in ('cpu', 'cuda'):
        raise ValueError('device must be cpu or cuda')
    from run import verify_ready, lock_held
    RUN.mkdir(parents=True, exist_ok=True)
    with (RUN/'notebook_launch.lock').open('a') as handle:
        fcntl.flock(handle, fcntl.LOCK_EX)
        if (RUN/'complete.json').exists():
            return dict(mode='complete', message='전체 학습·검증·Test 완료')
        current = worker()
        if current or lock_held(RUN/'run.lock'):
            return dict(mode='attached', **(current or {}))
        protocol = verify_ready()
        from run import check_device
        check_device(device)
        if phase == 'evaluate':
            from salt_daily.training import require_trained_variant
            for selected in ([variant] if variant else protocol['variants']):
                require_trained_variant(protocol, selected)
        command = [sys.executable, '-u', str(ROOT/'run.py'), phase, '--device', device]
        if variant is not None:
            command += ['--variant', variant]
        (ROOT/'logs').mkdir(exist_ok=True)
        with (ROOT/'logs/run.log').open('ab') as output:
            child = subprocess.Popen(command, cwd=ROOT,
                stdin=subprocess.DEVNULL, stdout=output, stderr=subprocess.STDOUT, start_new_session=True)
        _CHILDREN[child.pid] = child
        for _ in range(50):
            if child.poll() is not None:
                raise RuntimeError('Worker exited; inspect logs/run.log')
            current = identity(child.pid)
            if current:
                write_json(RUN/'notebook_launch.json', current)
                return dict(mode='started', **current)
            time.sleep(.1)
        raise RuntimeError('Process identity not yet available; check status before retrying')


def request_stop():
    current = worker()
    if not current or identity(current['pid']) != current:
        raise RuntimeError('실행 중인 이 실험의 프로세스가 없습니다.')
    os.kill(current['pid'], signal.SIGTERM)
    return '중단 요청: Train 논리 배치 또는 평가 저장 구간 이후 멈춥니다.'


def last_progress(folder):
    current = read(folder/'status.json', {})
    if current.get('event') in ('training_progress', 'evaluation_progress', 'aggregating_metrics', 'training_started'):
        return current
    path = folder/'events.jsonl'
    if not path.exists():
        return {}
    with path.open('rb') as handle:
        size = handle.seek(0, 2); handle.seek(max(0, size-256_000))
        if size > 256_000:
            handle.readline()
        lines = handle.read().decode().splitlines()
    for line in reversed(lines):
        try:
            item = json.loads(line)
            if item.get('event') in ('training_progress', 'evaluation_progress', 'aggregating_metrics', 'training_started'):
                return item
        except json.JSONDecodeError:
            pass
    return {}


def snapshot():
    protocol = read_json(ROOT/'protocol.json'); result = []
    counts = {name: item['rows'] for name, item in protocol['cohorts'].items()}
    epochs = protocol['training']['epochs_primary']
    for variant in protocol['variants']:
        folder = RUN/'models'/variant
        history = read(folder/'history.json', [])
        complete = read(folder/'complete.json', {})
        done_epochs = min(epochs, max([h['epoch'] for h in history]+[complete.get('epochs', 0)]))
        progress = last_progress(folder)
        amounts = dict(train_gnn=0, monitor=0, validation=0, test=0)
        done = done_epochs*(counts['train_gnn']+counts['monitor'])
        if done_epochs < epochs and progress.get('epoch') == done_epochs+1:
            if progress.get('event') in ('training_progress', 'training_started') and progress.get('phase') == 'train':
                amounts['train_gnn'] = min(counts['train_gnn'], progress.get('requested', 0))
            elif progress.get('scope') == 'monitor' or progress.get('phase') == 'monitor':
                amounts['train_gnn'] = counts['train_gnn']
                amounts['monitor'] = min(counts['monitor'], progress.get('scored', 0))
            done += amounts['train_gnn']+amounts['monitor']
        elif done_epochs == epochs:
            amounts['train_gnn'], amounts['monitor'] = counts['train_gnn'], counts['monitor']
        metrics = {}
        for scope in ('validation', 'test'):
            metrics[scope] = read(folder/f'{scope}.json', {})
            if metrics[scope]:
                amounts[scope] = counts[scope]
            elif progress.get('scope') == scope:
                amounts[scope] = min(counts[scope], progress.get('scored', 0))
            done += amounts[scope]
        result.append(dict(variant=variant, completed_epochs=done_epochs, counts=amounts,
            progress=progress, history=history, metrics=metrics, done=done,
            total=epochs*(counts['train_gnn']+counts['monitor'])+counts['validation']+counts['test']))
    from run import lock_held
    state = 'complete' if (RUN/'complete.json').exists() else ('running' if lock_held(RUN/'run.lock') else 'paused')
    if state == 'paused' and worker():
        state = 'starting'
    return dict(state=state, models=result, counts=counts, epochs=epochs,
                gpu_microbatch=protocol['training']['microbatch_max'],
                logical_batch=protocol['training']['logical_batch'],
                done=sum(m['done'] for m in result), total=sum(m['total'] for m in result),
                status=read(RUN/'status.json', {}))


def progress_html(value):
    def bar(label, done, total):
        ratio = done/total*100 if total else 0
        return (f'<p>{escape(label)}: <b>{done:,}/{total:,} ({ratio:.2f}%)</b><br>'
                f'<progress style="width:100%;height:20px" value="{done}" max="{total}"></progress></p>')
    result = f'<div style="max-width:950px"><b>SALT-GNN 직접 공유 탐색 · 결제 유형·Port · {escape(value["state"])}</b>'
    result += f'<p>GPU 배치 최대 {value["gpu_microbatch"]} · 논리 배치 {value["logical_batch"]} · 거래 입력 24차원</p>'
    result += bar('전체 처리 건수 (시간 비율 아님)', value['done'], value['total'])
    for model in value['models']:
        result += f'<hr><b>{model["variant"]} · {model["completed_epochs"]}/{value["epochs"]} 에폭 완료</b>'
        if model['metrics'].get('validation') and model['metrics'].get('test'):
            result += '<p><b>이 모델의 전체 Val·Test 완료</b></p>'
        for key, label in [('train_gnn', '현재 에폭 Train'), ('monitor', '현재 에폭 Monitor'),
                           ('validation', '최종 Val'), ('test', '전체 Test')]:
            result += bar(label, model['counts'][key], value['counts'][key])
        progress = model['progress']
        result += f'<p>최근 기록(UTC): {escape(progress.get("utc", "—"))}</p>'
        seconds = progress.get('train_seconds', progress.get('seconds', 0))
        if progress.get('batch_wall_seconds'):
            rate = progress['batch_targets']/progress['batch_wall_seconds']
            result += f'<p>최근 배치 처리 속도 {rate:,.1f}건/초 · 대상일 {escape(progress.get("day", "—"))}</p>'
        elif seconds:
            count = progress.get('scored', progress.get('requested', 0))
            result += f'<p>누적 평균 처리 속도 {count/seconds:,.1f}건/초 · 대상일 {escape(progress.get("day", "—"))}</p>'
    return result+'<p>다운샘플링·IPW 없음. 평가 지표는 완료된 전체 거래의 실제 건수로 계산합니다.</p></div>'


def show():
    from IPython.display import display, HTML
    value = snapshot(); display(HTML(progress_html(value))); return value


def watch(refresh_seconds=3, max_seconds=None):
    from IPython.display import display, HTML
    handle = display(HTML('진행률 읽는 중'), display_id=True); begin = time.monotonic()
    try:
        while True:
            value = snapshot(); handle.update(HTML(progress_html(value)))
            if value['state'] in ('complete', 'paused') or (max_seconds is not None and time.monotonic()-begin >= max_seconds):
                return value
            time.sleep(max(1, refresh_seconds))
    except KeyboardInterrupt:
        print('표시만 종료했습니다. 작업 중단은 중단 셀을 실행하세요.')
        return snapshot()
'''


In [ ]:
START_TRAINING = False
if START_TRAINING:
    from notebook_control import start_or_attach
    print(start_or_attach(device=DEVICE, phase="run"))


In [ ]:
if (WORK_DIR/"protocol.json").exists():
    from notebook_control import show
    state = show()
else:
    print("02 학습 입력 준비가 아직 완료되지 않았습니다.")


In [ ]:
WATCH_PROGRESS = START_TRAINING  # 실행 직후 자동 표시; 다시 연결하려면 True
if WATCH_PROGRESS:
    from notebook_control import watch
    state = watch(refresh_seconds=3, max_seconds=300)


In [ ]:
STOP_TRAINING = False
if STOP_TRAINING:
    from notebook_control import request_stop
    print(request_stop())


`last.pt`는 모델·Adam·난수·대상 진행 위치, `best.pt`는 Monitor로 고른 모델입니다.
모델별 `complete.json`은 학습 완료, `evaluation_complete.json`은 개별 Val/Test 완료를 뜻합니다.
03을 다시 시작하면 이미 완료한 단계는 재사용합니다. 04에서 A3만 개별 평가할 수도 있으며 동일 worker lock으로 중복 실행을 막습니다.
로그는 `WORK_DIR/logs/run.log`, 결과는 `WORK_DIR/runs/daily_close_001/models/<모델>/`입니다.
